# PI1M Pretrain -> Finetune (Encoder Only / Decoder Only)

- 목적: `PI1M` p-SMILES 코퍼스로 문법/패턴을 사전학습한 뒤, 현재 전도도 조건 토큰(`LOW/MID/HIGH`) 데이터셋으로 미세조정
- 대상 모델: `Encoder_Only`, `Encoder_Only` (둘 중 하나 선택)
- 현재 `Encoder_Only/utils/dataloader.py`의 설정(quantile 3분할, 전체데이터 학습)은 그대로 사용


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import math
import time
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def _is_models_root(p: Path) -> bool:
    return (p / "utils").is_dir() and (p / "models").is_dir() and (p / "notebooks").is_dir() and (p / "data").is_dir()


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if _is_models_root(p):
            return p
        cand = p / "MY_PAPER_RELATED" / "MODELS"
        if _is_models_root(cand):
            return cand
    raise FileNotFoundError("Could not locate MODELS project root")
PROJECT_ROOT = resolve_project_root()
os.environ["MODELS_VARIANT"] = "Encoder_Only"
WORKSPACE_ROOT = PROJECT_ROOT.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('WORKSPACE_ROOT =', WORKSPACE_ROOT)
print('CUDA available =', torch.cuda.is_available())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
from utils.utils import *
from utils.dataloader import dataset as ft_dataset, eval_dataloader as ft_train_loader
from utils.pi1m_pretrain import build_pi1m_pretrain_loader
from models.Trans import Encoder_Only

index_to_token_ft = {idx: tok for tok, idx in ft_dataset.vocab.items()}
print('Finetune dataset size:', len(ft_dataset))
print('Finetune vocab size:', ft_dataset.vocab_size)
print('Condition mode: scalar (normalized conductivity)')
print('Property mean/std (normalized space):', float(ft_dataset.properties.mean()), float(ft_dataset.properties.std()))


In [ ]:
# ==== Config ====
MODE = 'Encoder_Only_SELFIES_PSMILES'
PI1M_USE_V2 = True             # True -> PI1M_v2.csv, False -> PI1M.csv
PI1M_MAX_ROWS = None           # None = full PI1M (~1M). 빠른 테스트는 예: 200_000
PI1M_BATCH_SIZE = 256
FINETUNE_BATCH_SIZE_INFO = 'uses existing ft_train_loader (full dataset)'

PRETRAIN_EPOCHS = 1            # full PI1M는 1~3부터 시작 권장
FINETUNE_EPOCHS = 200          # 필요시 조정
PRETRAIN_LR = 3e-4
FINETUNE_LR = 3e-4
WEIGHT_DECAY = 0.0

SAVE_DIR = PROJECT_ROOT / 'checkpoints'
PRETRAIN_SAVE_DIR = SAVE_DIR / 'pretrain'
PRETRAIN_SAVE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

PI1M_CSV = WORKSPACE_ROOT / 'PI1M' / ('PI1M_v2.csv' if PI1M_USE_V2 else 'PI1M.csv')
assert PI1M_CSV.exists(), f'PI1M CSV not found: {PI1M_CSV}'

print('MODE =', MODE)
print('PI1M_CSV =', PI1M_CSV)
print('PI1M_MAX_ROWS =', PI1M_MAX_ROWS)
print('PRETRAIN_EPOCHS =', PRETRAIN_EPOCHS, 'FINETUNE_EPOCHS =', FINETUNE_EPOCHS)


In [ ]:
# ==== Build PI1M pretrain dataloader ====
pi1m_cache_dir = PROJECT_ROOT / 'checkpoints' / 'cache'
pi1m_ds, pi1m_loader = build_pi1m_pretrain_loader(
    csv_path=PI1M_CSV,
    batch_size=PI1M_BATCH_SIZE,
    cache_dir=pi1m_cache_dir,
    max_rows=PI1M_MAX_ROWS,
    smiles_col='SMILES',
    shuffle=True,
)

print('PI1M rows read:', getattr(pi1m_ds, 'num_rows_read', 'n/a'))
print('PI1M valid rows:', getattr(pi1m_ds, 'num_rows_valid', len(pi1m_ds)))
print('PI1M vocab size:', pi1m_ds.vocab_size)
print('PI1M max_len:', pi1m_ds.max_len)
print('PI1M cond scalar example:', float(getattr(pi1m_ds, 'properties', torch.zeros(1,1,1))[0].view(-1)[0]))
print('Finetune max_len:', ft_dataset.max_len)
print('NOTE: pretrain model vocab != finetune vocab is allowed (remap helper below).')


In [ ]:
# ==== Model / loading helpers ====
MODEL_CLS = Encoder_Only
MODE_STEM = 'encoder_only'

def build_model(vocab_size=None):
    return MODEL_CLS(vocab_size=vocab_size).to(device)

def train_one_epoch(model, loader, optimizer, loss_fn, pad_idx, vocab_size, use_tqdm=False):
    model.train()
    total = 0.0
    n_steps = 0
    loader = tqdm(loader) if use_tqdm else loader
    for batch in loader:
        _, sm_dec_in, sm_dec_out, cond_scalar = [t.to(device) for t in batch]
        optimizer.zero_grad(set_to_none=True)
        attn_mask = (sm_dec_in == pad_idx)
        logits = model(sm_dec_in, cond_scalar, attn_mask=attn_mask)
        loss = loss_fn(logits.reshape(-1, vocab_size), sm_dec_out.view(-1))
        loss.backward()
        optimizer.step()
        total += float(loss.item())
        n_steps += 1
    return total / max(n_steps, 1)

def remap_token_layers_(model, src_state, src_vocab, dst_vocab):
    copied = 0
    missed = 0
    with torch.no_grad():
        # embedding
        if 'smi_emb.weight' in src_state:
            src_emb = src_state['smi_emb.weight']
            dst_emb = model.smi_emb.weight
            for tok, dst_idx in dst_vocab.items():
                src_idx = src_vocab.get(tok)
                if src_idx is None or src_idx >= src_emb.size(0) or dst_idx >= dst_emb.size(0):
                    missed += 1
                    continue
                dst_emb[dst_idx].copy_(src_emb[src_idx].to(dst_emb.device))
                copied += 1
        # output head
        if 'predict.weight' in src_state and 'predict.bias' in src_state:
            src_w = src_state['predict.weight']
            src_b = src_state['predict.bias']
            dst_w = model.predict.weight
            dst_b = model.predict.bias
            for tok, dst_idx in dst_vocab.items():
                src_idx = src_vocab.get(tok)
                if src_idx is None:
                    continue
                if src_idx < src_w.size(0) and dst_idx < dst_w.size(0):
                    dst_w[dst_idx].copy_(src_w[src_idx].to(dst_w.device))
                if src_idx < src_b.size(0) and dst_idx < dst_b.size(0):
                    dst_b[dst_idx].copy_(src_b[src_idx].to(dst_b.device))
    return copied, missed

def load_pretrain_with_vocab_remap(model, ckpt_path, dst_vocab):
    blob = torch.load(ckpt_path, map_location='cpu')
    src_state = blob['state_dict'] if isinstance(blob, dict) and 'state_dict' in blob else blob
    src_vocab = blob.get('vocab') if isinstance(blob, dict) else None
    if src_vocab is None:
        raise ValueError('Checkpoint missing vocab for remap.')

    current = model.state_dict()
    matched = {}
    skipped = []
    token_layer_keys = {'smi_emb.weight', 'predict.weight', 'predict.bias'}
    for k, v in src_state.items():
        if k in token_layer_keys:
            continue
        if k in current and tuple(current[k].shape) == tuple(v.shape):
            matched[k] = v
        else:
            skipped.append(k)
    model.load_state_dict(matched, strict=False)
    copied, missed = remap_token_layers_(model, src_state, src_vocab, dst_vocab)
    overlap = sum(1 for tok in dst_vocab if tok in src_vocab)
    print(f'Base params loaded: matched={len(matched)} skipped={len(skipped)}')
    print(f'Token remap overlap: {overlap}/{len(dst_vocab)} (embedding rows copied={copied}, missed={missed})')
    if skipped:
        print('Skipped examples:', skipped[:10])
    return blob


In [ ]:
# ==== Stage 1: PI1M pretraining ====
pre_model = build_model(vocab_size=pi1m_ds.vocab_size)
pre_pad_idx = pi1m_ds.vocab['[PAD]']
pre_loss_fn = nn.CrossEntropyLoss(reduction='mean', ignore_index=pre_pad_idx)
pre_optim = AdamW(pre_model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)

pretrain_losses = []
pre_ckpt_path = PRETRAIN_SAVE_DIR / f'{MODE_STEM}_{MODE}_pi1m_pretrain.pth'

for ep in tqdm(range(1, PRETRAIN_EPOCHS + 1), desc='PI1M pretrain epochs'):
    t0 = time.time()
    loss = train_one_epoch(pre_model, pi1m_loader, pre_optim, pre_loss_fn, pre_pad_idx, pi1m_ds.vocab_size, True)
    pretrain_losses.append(loss)
    dt = time.time() - t0
    print(f'[Pretrain][{ep}/{PRETRAIN_EPOCHS}] loss={loss:.6f} time={dt/60:.2f} min')

torch.save({
    'mode': MODE,
    'state_dict': pre_model.state_dict(),
    'vocab': pi1m_ds.vocab,
    'vocab_size': pi1m_ds.vocab_size,
    'max_len': pi1m_ds.max_len,
    'pi1m_csv': str(PI1M_CSV),
    'pi1m_max_rows': PI1M_MAX_ROWS,
    'pretrain_epochs': PRETRAIN_EPOCHS,
    'pretrain_losses': pretrain_losses,
}, pre_ckpt_path)
print('Saved pretrain checkpoint ->', pre_ckpt_path)


In [ ]:
# ==== Stage 2: finetune on current scalar-conductivity dataset (full dataset) ====
ft_model = build_model(vocab_size=ft_dataset.vocab_size)
_ = load_pretrain_with_vocab_remap(ft_model, pre_ckpt_path, ft_dataset.vocab)

ft_pad_idx = ft_dataset.vocab['[PAD]']
ft_loss_fn = nn.CrossEntropyLoss(reduction='mean', ignore_index=ft_pad_idx)
ft_optim = AdamW(ft_model.parameters(), lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY)

finetune_losses = []
for ep in tqdm(range(1, FINETUNE_EPOCHS + 1), desc='Finetune epochs'):
    t0 = time.time()
    loss = train_one_epoch(ft_model, ft_train_loader, ft_optim, ft_loss_fn, ft_pad_idx, ft_dataset.vocab_size)
    finetune_losses.append(loss)
    dt = time.time() - t0
    print(f'[Finetune][{ep}/{FINETUNE_EPOCHS}] loss={loss:.6f} time={dt:.1f}s')

final_ckpt_path = SAVE_DIR / f'{MODE_STEM}_{MODE}_pi1m_finetuned.pth'
torch.save(ft_model.state_dict(), final_ckpt_path)
print('Saved finetuned weights ->', final_ckpt_path)


In [ ]:
# Optional: save finetune checkpoint with metadata (recommended for reproducibility)
ft_bundle_path = SAVE_DIR / f'{MODE_STEM}_{MODE}_pi1m_finetuned_bundle.pth'
torch.save({
    'mode': MODE,
    'state_dict': ft_model.state_dict(),
    'vocab': ft_dataset.vocab,
    'condition_mode': 'scalar',
    'pretrain_ckpt': str(pre_ckpt_path),
    'pretrain_epochs': PRETRAIN_EPOCHS,
    'finetune_epochs': FINETUNE_EPOCHS,
    'finetune_losses': finetune_losses,
}, ft_bundle_path)
print('Saved finetune bundle ->', ft_bundle_path)

print('Last 5 pretrain losses:', pretrain_losses[-5:])
print('Last 5 finetune losses:', finetune_losses[-5:])


## Notes

- `PI1M_MAX_ROWS=None`는 메모리/시간 부담이 크다. 첫 실행은 `100000~300000`으로 캐시 생성 후 점진적으로 늘리는 게 안전하다.
- pretrain checkpoint에는 `PI1M vocab`이 저장되고, finetune 단계에서 공통 토큰 임베딩/출력층을 `token string` 기준으로 remap한다.
- 현재 finetune 데이터는 `quantile 3분할 [LOW/MID/HIGH] + 전체데이터 학습` 설정을 그대로 사용한다.
